# Capture an inference baseline with SageMaker Model Monitor, generate a deliberately drifted payload, and prove the monitor surfaces a violation

A production fraud model has been silently producing increasingly bad predictions for two months and nobody noticed because nothing alarmed on the inference traffic itself. The fix is SageMaker Model Monitor: a Processing Job that compares incoming inference inputs against a baseline statistics file and surfaces violations to CloudWatch.

You have one session to retrofit Model Monitor onto a Real-Time endpoint, capture a baseline from training data, generate 100 well-behaved inferences and 100 deliberately drifted inferences, wait for the first scheduled monitoring run, and prove the Monitor finds the drift.

Five artifacts ship in this lab:

- A Real-Time SageMaker endpoint on ml.t2.medium with `DataCaptureConfig` enabled (100 percent capture of inputs and outputs to S3).
- A baseline Processing Job that produces `statistics.json` and `constraints.json` from the training distribution.
- A Monitoring Schedule that fires every 15 minutes (the lab does not wait an hour) against the baseline plus the captured traffic.
- 100 baseline inferences plus 100 drifted inferences captured to the DataCapture prefix.
- The first scheduled monitoring run reporting `CompletedWithViolations` with the drifted feature surfaced in `constraint_violations.json`.

**Compare-it lab.** This is a Compare-it lab. The same endpoint serves baseline traffic and drifted traffic; the comparison (baseline statistics vs drifted statistics, which is what Model Monitor flags) lives in the monitoring output. Checkpoint 5 captures the drifted feature name and violation count into a `comparisonMetric` string the Option D Colab card surfaces on PASS. The validator confirms the pipeline produced the violation; it does not compute the drift magnitude itself.

**Time.** Plan for about 60 minutes hands-on. Training plus endpoint create plus baseline job plus traffic generation is about 25 minutes. The first scheduled monitoring run takes 5 to 20 minutes after the schedule fires. Cleanup is another 5 minutes (stop schedule before delete endpoint). Budget 120 minutes if the schedule misses its first window.

**Cost.** This lab costs about 10 to 20 cents per session. The Real-Time endpoint at $0.07 per hour for 60 minutes is the line item. Two Processing Jobs (baseline + first scheduled monitor run) on ml.m5.xlarge at $0.230 per hour for 5 minutes each add about 4 cents. DataCapture S3 puts are fractions of a penny. CloudWatch metrics are free at this volume.

The trap here is the monitoring schedule. If you delete the endpoint without first stopping the schedule, the schedule keeps firing every 15 minutes against an endpoint that does not exist anymore. Each failed monitoring run is still a Processing Job that bills two cents. Multiply by 96 runs a day and the orphaned schedule costs $2 a day until you notice. Cleanup stops the schedule first, deletes the schedule, then deletes the endpoint.

This lab maps to AWS MLA-C01 Domain 4 (ML Solution Monitoring, Maintenance, and Security, 24%) on SageMaker Model Monitor (data quality, model quality, bias drift, feature attribution drift) and the cleanup-ordering trap that the exam tests directly.


In [ ]:
# NBVAL_SKIP
# Install labrun-checks plus the SageMaker SDK.

!pip install --quiet labrun-checks==0.6.0 sagemaker==2.232.0

In [ ]:
# Imports and per-lab constants. Resource names follow the
# labrun-{lab-slug}-{descriptor} pattern.

import atexit
import csv
import getpass
import io
import json
import random
import sys
import time
from datetime import datetime, timedelta, timezone

import boto3
from botocore.exceptions import ClientError

from labrun_checks import (
    register,
    check,
    cleanup,
    run_cleanup,
    CleanupResource,
    CheckpointResult,
)

LAB_ID = "model-monitor-data-drift"
LAB_TAG_KEY = "labrun:lab-slug"
LAB_TAG_VALUE = LAB_ID
REGION = "us-east-1"

MONITOR_ROLE_NAME = f"labrun-{LAB_ID}-role"
MONITOR_INLINE_POLICY = "labrun-mla-lab11-monitor-policy"
TRAINING_JOB = f"labrun-{LAB_ID}-train"
MODEL_NAME = f"labrun-{LAB_ID}-model"
ENDPOINT_CONFIG_NAME = f"labrun-{LAB_ID}-config"
ENDPOINT_NAME = f"labrun-{LAB_ID}-endpoint"
BASELINE_JOB = f"labrun-{LAB_ID}-baseline"
SCHEDULE_NAME = f"labrun-{LAB_ID}-schedule"
DRIFTED_FEATURE_INDEX = 1  # the column index we will deliberately drift

BUCKET_NAME = None
ACCOUNT_ID = None
MONITOR_ROLE_ARN = None
XGBOOST_IMAGE_URI = None
MODEL_ARTIFACT = None
DRIFTED_FEATURE_NAME = f"f{DRIFTED_FEATURE_INDEX}"
VIOLATION_COUNT = 0
SESSION_START = datetime.now(timezone.utc)

In [ ]:
# NBVAL_SKIP
# Register the session, validate AWS credentials via STS GetCallerIdentity,
# resolve account-derived names.

session_token = getpass.getpass("Paste your session token from labrun.dev: ")
aws_access_key_id = getpass.getpass("AWS_ACCESS_KEY_ID: ")
aws_secret_access_key = getpass.getpass("AWS_SECRET_ACCESS_KEY: ")
aws_session_token = getpass.getpass(
    "AWS_SESSION_TOKEN (leave blank for long-lived credentials): "
).strip()
region_input = input(f"AWS region (default {REGION}): ").strip()
if region_input and region_input != REGION:
    print(f"Region {region_input!r} is not supported for this lab.")
    print(f"MLA Batch 3 labs run in {REGION} (N. Virginia).")
    print("Re-run this cell and accept the default.")
    raise SystemExit(1)

AWS_CREDENTIALS = {
    "aws_access_key_id": aws_access_key_id,
    "aws_secret_access_key": aws_secret_access_key,
    "region": REGION,
}
if aws_session_token:
    AWS_CREDENTIALS["aws_session_token"] = aws_session_token

sts = boto3.client(
    "sts",
    aws_access_key_id=aws_access_key_id,
    aws_secret_access_key=aws_secret_access_key,
    aws_session_token=aws_session_token or None,
    region_name=REGION,
)
try:
    identity = sts.get_caller_identity()
except ClientError as e:
    print("Credentials are missing or expired. STS GetCallerIdentity failed:")
    print(f"  {e}")
    print()
    print("Refresh your AWS credentials and re-run this cell.")
    raise SystemExit(1)

ACCOUNT_ID = identity["Account"]
CALLER_ARN = identity["Arn"]
print(f"Credentials validated. Account: {ACCOUNT_ID}")
print(f"Caller ARN: {CALLER_ARN}")
print(f"Region: {REGION}")

BUCKET_NAME = f"labrun-{LAB_ID}-{ACCOUNT_ID}"
MONITOR_ROLE_ARN = f"arn:aws:iam::{ACCOUNT_ID}:role/{MONITOR_ROLE_NAME}"
print(f"Bucket: {BUCKET_NAME}")
print(f"Monitor role ARN: {MONITOR_ROLE_ARN}")

register(session_token=session_token, lab_id=LAB_ID, credentials=AWS_CREDENTIALS)
print(f"Session registered for lab_id: {LAB_ID}")

In [ ]:
# NBVAL_SKIP
# CLEANUP_MANIFEST + atexit + orphan scan. labrun-checks 0.6.0 has no
# adapters for SageMaker endpoints, configs, models, monitoring schedules,
# or processing jobs; the cleanup cell tears those down manually BEFORE
# run_cleanup walks the manifest. The manifest below contains only
# adapter-supported types.
#
# Critical resources: the Real-Time endpoint ($0.07/hour ml.t2.medium) and
# the Monitoring Schedule (which keeps firing Processing Jobs against the
# endpoint until stopped, billing $0.02 per fired run).

CLEANUP_MANIFEST = [
    CleanupResource(
        type="iam_role",
        id=MONITOR_ROLE_NAME,
        region=REGION,
        cli_delete_command=f"aws iam delete-role --role-name {MONITOR_ROLE_NAME}",
    ),
    CleanupResource(
        type="s3_bucket",
        id=BUCKET_NAME,
        region=REGION,
        cli_delete_command=f"aws s3 rb s3://{BUCKET_NAME} --force",
    ),
]


def _atexit_cleanup() -> None:
    try:
        run_cleanup(CLEANUP_MANIFEST)
    except Exception:
        pass


atexit.register(_atexit_cleanup)


def scan_for_orphans() -> list:
    client = boto3.client(
        "resourcegroupstaggingapi",
        aws_access_key_id=AWS_CREDENTIALS["aws_access_key_id"],
        aws_secret_access_key=AWS_CREDENTIALS["aws_secret_access_key"],
        aws_session_token=AWS_CREDENTIALS.get("aws_session_token"),
        region_name=REGION,
    )
    arns = []
    paginator = client.get_paginator("get_resources")
    for page in paginator.paginate(
        TagFilters=[{"Key": LAB_TAG_KEY, "Values": [LAB_TAG_VALUE]}],
    ):
        for item in page.get("ResourceTagMappingList", []):
            arns.append(item.get("ResourceARN", ""))
    return arns


orphans = scan_for_orphans()
if orphans:
    print(f"BLOCKED: existing resources tagged labrun:lab-slug={LAB_TAG_VALUE} were found:")
    for arn in orphans:
        print("  -", arn)
    print()
    print("Run the cleanup cell at the bottom of this notebook first, or remove")
    print("them manually with the AWS CLI commands the cleanup cell prints.")
    raise SystemExit(1)

print("No prior orphans found. Safe to create resources for this session.")

## Task 1: Train a small XGBoost model and deploy it to a Real-Time endpoint with DataCapture enabled

Train a small XGBoost model on a deterministic 1000-row binary classification dataset (8 features, signal in feature 0). The trained model lives at the S3 path SageMaker returns. Deploy it to a Real-Time endpoint on ml.t2.medium with `DataCaptureConfig.EnableCapture=True` and `InitialSamplingPercentage=100`. The capture writes every input plus output to the S3 capture prefix.

Build it in this order:

1. Create the bucket and the monitor role.
2. Generate `training.csv` (1000 rows) and upload.
3. Train the XGBoost model and capture `MODEL_ARTIFACT`.
4. Create the SageMaker Model, the endpoint config (with DataCaptureConfig), and the endpoint.
5. Wait for the endpoint to reach `InService`.

Your job: write the endpoint config and the endpoint create calls. The bucket, role, training, model build, and capture config are pre-wired.

In [ ]:
# NBVAL_SKIP
# Task 1: train and deploy with DataCapture.

import sagemaker

s3 = boto3.client(
    "s3",
    aws_access_key_id=AWS_CREDENTIALS["aws_access_key_id"],
    aws_secret_access_key=AWS_CREDENTIALS["aws_secret_access_key"],
    aws_session_token=AWS_CREDENTIALS.get("aws_session_token"),
    region_name=REGION,
)
iam = boto3.client(
    "iam",
    aws_access_key_id=AWS_CREDENTIALS["aws_access_key_id"],
    aws_secret_access_key=AWS_CREDENTIALS["aws_secret_access_key"],
    aws_session_token=AWS_CREDENTIALS.get("aws_session_token"),
    region_name=REGION,
)
sm = boto3.client(
    "sagemaker",
    aws_access_key_id=AWS_CREDENTIALS["aws_access_key_id"],
    aws_secret_access_key=AWS_CREDENTIALS["aws_secret_access_key"],
    aws_session_token=AWS_CREDENTIALS.get("aws_session_token"),
    region_name=REGION,
)
sm_runtime = boto3.client(
    "sagemaker-runtime",
    aws_access_key_id=AWS_CREDENTIALS["aws_access_key_id"],
    aws_secret_access_key=AWS_CREDENTIALS["aws_secret_access_key"],
    aws_session_token=AWS_CREDENTIALS.get("aws_session_token"),
    region_name=REGION,
)

try:
    s3.create_bucket(Bucket=BUCKET_NAME)
    print(f"Created bucket: {BUCKET_NAME}")
except ClientError as e:
    if e.response["Error"]["Code"] in ("BucketAlreadyOwnedByYou", "BucketAlreadyExists"):
        print(f"Bucket {BUCKET_NAME} already exists, reusing.")
    else:
        raise
s3.put_bucket_tagging(
    Bucket=BUCKET_NAME,
    Tagging={"TagSet": [{"Key": LAB_TAG_KEY, "Value": LAB_TAG_VALUE}]},
)

# Generate training distribution. 8 features, signal in feature 0.
random.seed(11)
header = ["label"] + [f"f{i}" for i in range(8)]
buf = io.StringIO()
w = csv.writer(buf)
w.writerow(header)
for _ in range(1000):
    f0 = random.gauss(0, 1)
    feats = [f0] + [random.gauss(0, 1) for _ in range(7)]
    label = 1 if (f0 + random.gauss(0, 0.3)) > 0 else 0
    w.writerow([label] + [f"{v:.4f}" for v in feats])
training_csv_with_header = buf.getvalue()
training_csv_no_header = "\n".join(training_csv_with_header.splitlines()[1:]) + "\n"
s3.put_object(
    Bucket=BUCKET_NAME, Key="input/training.csv", Body=training_csv_with_header.encode("utf-8"),
)
s3.put_object(
    Bucket=BUCKET_NAME, Key="input/train_noheader.csv", Body=training_csv_no_header.encode("utf-8"),
)
print("Uploaded training.csv (with header) and train_noheader.csv (for XGBoost).")

trust = {
    "Version": "2012-10-17",
    "Statement": [
        {
            "Effect": "Allow",
            "Principal": {"Service": "sagemaker.amazonaws.com"},
            "Action": "sts:AssumeRole",
        }
    ],
}
try:
    iam.create_role(
        RoleName=MONITOR_ROLE_NAME,
        AssumeRolePolicyDocument=json.dumps(trust),
        Description="labrun mla lab 11 monitor role",
        Tags=[{"Key": LAB_TAG_KEY, "Value": LAB_TAG_VALUE}],
    )
    print(f"Created role: {MONITOR_ROLE_NAME}")
except ClientError as e:
    if e.response["Error"]["Code"] == "EntityAlreadyExists":
        print(f"Role {MONITOR_ROLE_NAME} already exists, reusing.")
    else:
        raise

policy = {
    "Version": "2012-10-17",
    "Statement": [
        {
            "Effect": "Allow",
            "Action": ["s3:GetObject", "s3:PutObject", "s3:DeleteObject", "s3:ListBucket"],
            "Resource": [
                f"arn:aws:s3:::{BUCKET_NAME}",
                f"arn:aws:s3:::{BUCKET_NAME}/*",
            ],
        },
        {
            "Effect": "Allow",
            "Action": ["logs:CreateLogGroup", "logs:CreateLogStream", "logs:PutLogEvents"],
            "Resource": "*",
        },
        {
            "Effect": "Allow",
            "Action": ["cloudwatch:PutMetricData"],
            "Resource": "*",
        },
    ],
}
iam.put_role_policy(
    RoleName=MONITOR_ROLE_NAME,
    PolicyName=MONITOR_INLINE_POLICY,
    PolicyDocument=json.dumps(policy),
)
print(f"Attached inline policy {MONITOR_INLINE_POLICY}.")
print("Your IAM role is stuck in traffic, give it 10 seconds...")
time.sleep(10)

XGBOOST_IMAGE_URI = sagemaker.image_uris.retrieve(
    framework="xgboost", region=REGION, version="1.5-1"
)
print(f"XGBoost image: {XGBOOST_IMAGE_URI}")

sm.create_training_job(
    TrainingJobName=TRAINING_JOB,
    AlgorithmSpecification={
        "TrainingImage": XGBOOST_IMAGE_URI,
        "TrainingInputMode": "File",
    },
    RoleArn=MONITOR_ROLE_ARN,
    InputDataConfig=[
        {
            "ChannelName": "train",
            "DataSource": {
                "S3DataSource": {
                    "S3DataType": "S3Prefix",
                    "S3Uri": f"s3://{BUCKET_NAME}/input/train_noheader.csv",
                    "S3DataDistributionType": "FullyReplicated",
                }
            },
            "ContentType": "text/csv",
        }
    ],
    OutputDataConfig={"S3OutputPath": f"s3://{BUCKET_NAME}/output/train/"},
    ResourceConfig={"InstanceType": "ml.m5.large", "InstanceCount": 1, "VolumeSizeInGB": 10},
    StoppingCondition={"MaxRuntimeInSeconds": 900},
    HyperParameters={
        "objective": "binary:logistic",
        "num_round": "30",
        "max_depth": "4",
        "eta": "0.2",
        "eval_metric": "auc",
    },
    Tags=[{"Key": LAB_TAG_KEY, "Value": LAB_TAG_VALUE}],
)
print("Training the model, give it about 5 minutes...")
elapsed = 0
while elapsed < 900:
    resp = sm.describe_training_job(TrainingJobName=TRAINING_JOB)
    status = resp["TrainingJobStatus"]
    if status in ("Completed", "Failed", "Stopped"):
        break
    time.sleep(15)
    elapsed += 15
    if elapsed % 60 == 0:
        print(f"  {elapsed}s elapsed, status: {status}")
if status != "Completed":
    print(f"Training ended with status {status}.")
    raise SystemExit(1)
MODEL_ARTIFACT = sm.describe_training_job(TrainingJobName=TRAINING_JOB)["ModelArtifacts"][
    "S3ModelArtifacts"
]
print(f"Model artifact: {MODEL_ARTIFACT}")

sm.create_model(
    ModelName=MODEL_NAME,
    ExecutionRoleArn=MONITOR_ROLE_ARN,
    PrimaryContainer={"Image": XGBOOST_IMAGE_URI, "ModelDataUrl": MODEL_ARTIFACT},
    Tags=[{"Key": LAB_TAG_KEY, "Value": LAB_TAG_VALUE}],
)
print(f"Created SageMaker Model: {MODEL_NAME}")

data_capture_config = {
    "EnableCapture": True,
    "InitialSamplingPercentage": 100,
    "DestinationS3Uri": f"s3://{BUCKET_NAME}/data-capture/",
    "CaptureOptions": [{"CaptureMode": "Input"}, {"CaptureMode": "Output"}],
}

# YOUR CODE: create the endpoint config using sm.create_endpoint_config()
# with EndpointConfigName=ENDPOINT_CONFIG_NAME, ProductionVariants=[{
#     "VariantName": "AllTraffic", "ModelName": MODEL_NAME,
#     "InitialInstanceCount": 1, "InstanceType": "ml.t2.medium",
# }], DataCaptureConfig=data_capture_config,
# Tags=[{"Key": LAB_TAG_KEY, "Value": LAB_TAG_VALUE}].
print(f"Created endpoint config: {ENDPOINT_CONFIG_NAME}")

# YOUR CODE: create the endpoint using sm.create_endpoint() with
# EndpointName=ENDPOINT_NAME, EndpointConfigName=ENDPOINT_CONFIG_NAME,
# Tags=[{"Key": LAB_TAG_KEY, "Value": LAB_TAG_VALUE}].
print(f"Creating endpoint: {ENDPOINT_NAME}")

print("Endpoint is provisioning ml.t2.medium, give it about 5 minutes...")
elapsed = 0
while elapsed < 900:
    resp = sm.describe_endpoint(EndpointName=ENDPOINT_NAME)
    status = resp["EndpointStatus"]
    if status in ("InService", "Failed"):
        break
    time.sleep(15)
    elapsed += 15
    if elapsed % 60 == 0:
        print(f"  {elapsed}s elapsed, status: {status}")
if status != "InService":
    print(f"Endpoint reached status {status}, not InService.")
    raise SystemExit(1)
print(f"Endpoint {ENDPOINT_NAME} is InService.")

# Generate 10 quick baseline-distribution inferences so DataCapture starts capturing.
sample_row = ",".join(["0.1"] * 8)
for _ in range(10):
    sm_runtime.invoke_endpoint(
        EndpointName=ENDPOINT_NAME,
        ContentType="text/csv",
        Body=sample_row.encode("utf-8"),
    )["Body"].read()
print("Sent 10 warm-up invocations.")
print("DataCapture is writing JSONL files every ~30 seconds, give it 90 seconds...")
time.sleep(90)

In [ ]:
# NBVAL_SKIP
# Checkpoint 1: endpoint InService, DataCapture enabled at 100%, at least
# one capture object present in S3.

def checkpoint_1(session):
    try:
        sm_c = boto3.client(
            "sagemaker",
            aws_access_key_id=AWS_CREDENTIALS["aws_access_key_id"],
            aws_secret_access_key=AWS_CREDENTIALS["aws_secret_access_key"],
            aws_session_token=AWS_CREDENTIALS.get("aws_session_token"),
            region_name=REGION,
        )
        s3_c = boto3.client(
            "s3",
            aws_access_key_id=AWS_CREDENTIALS["aws_access_key_id"],
            aws_secret_access_key=AWS_CREDENTIALS["aws_secret_access_key"],
            aws_session_token=AWS_CREDENTIALS.get("aws_session_token"),
            region_name=REGION,
        )
        ep = sm_c.describe_endpoint(EndpointName=ENDPOINT_NAME)
        if ep["EndpointStatus"] != "InService":
            return CheckpointResult(
                status="fail",
                error_reason=(
                    f"Endpoint {ENDPOINT_NAME} status is {ep['EndpointStatus']!r}, not InService."
                ),
            )
        cfg = sm_c.describe_endpoint_config(EndpointConfigName=ep["EndpointConfigName"])
        dcc = cfg.get("DataCaptureConfig", {})
        if not dcc.get("EnableCapture"):
            return CheckpointResult(
                status="fail",
                error_reason=(
                    "DataCaptureConfig.EnableCapture is False. Model Monitor cannot "
                    "work without capture; recreate the endpoint config with EnableCapture=True."
                ),
            )
        if dcc.get("InitialSamplingPercentage") != 100:
            return CheckpointResult(
                status="fail",
                error_reason=(
                    f"InitialSamplingPercentage is {dcc.get('InitialSamplingPercentage')}, "
                    f"expected 100 (the lab pins 100 percent capture)."
                ),
            )
        listed = s3_c.list_objects_v2(Bucket=BUCKET_NAME, Prefix="data-capture/")
        if not listed.get("Contents"):
            return CheckpointResult(
                status="fail",
                error_reason=(
                    "data-capture/ prefix is empty. Capture batches land every ~30 seconds; "
                    "wait 60 to 90 seconds after invocations and re-run the checkpoint."
                ),
            )
        return CheckpointResult(status="pass")
    except Exception as e:
        return CheckpointResult(status="error", error_reason=str(e))


check(1, checkpoint_1)

<details><summary>Hint 1 (nudge)</summary>

Two blanks: the endpoint config (with `DataCaptureConfig`) and the endpoint itself. The config dict and the ml.t2.medium choice are pinned for cost reasons.

</details>

<details><summary>Hint 2 (direction)</summary>

Pass `DataCaptureConfig=data_capture_config` to `create_endpoint_config`. The endpoint create only needs `EndpointName`, `EndpointConfigName`, and tags.

</details>

<details><summary>Hint 3 (spoiler)</summary>

Complete working solution for Task 1 (endpoint config + endpoint create):

```python
sm.create_endpoint_config(
    EndpointConfigName=ENDPOINT_CONFIG_NAME,
    ProductionVariants=[
        {
            "VariantName": "AllTraffic",
            "ModelName": MODEL_NAME,
            "InitialInstanceCount": 1,
            "InstanceType": "ml.t2.medium",
        }
    ],
    DataCaptureConfig=data_capture_config,
    Tags=[{"Key": LAB_TAG_KEY, "Value": LAB_TAG_VALUE}],
)

sm.create_endpoint(
    EndpointName=ENDPOINT_NAME,
    EndpointConfigName=ENDPOINT_CONFIG_NAME,
    Tags=[{"Key": LAB_TAG_KEY, "Value": LAB_TAG_VALUE}],
)
```

</details>

**Wallet check.** Training on ml.m5.large for 5 minutes is about $0.01. The endpoint on ml.t2.medium at $0.07/hour started billing the moment it went InService. The 10 warm-up invocations are fractions of a penny. Damage so far: about $0.02. Your coffee is fine.

## Task 2: Run the baseline Processing Job and write `statistics.json` plus `constraints.json` to S3

Model Monitor's baseline job runs a Processing Job over the training distribution that produced the model. The output is two JSON files: `statistics.json` (per-feature distribution stats) and `constraints.json` (per-feature constraints derived from the stats). Both files become the comparison reference for every scheduled monitoring run.

Use `DefaultModelMonitor` from `sagemaker.model_monitor`. Call `suggest_baseline` with the training-data S3 URI and the baseline-output S3 URI; the job runs on ml.m5.xlarge for about 5 minutes.

In [ ]:
# NBVAL_SKIP
# Task 2: baseline Processing Job.

from sagemaker.model_monitor import DefaultModelMonitor
from sagemaker.model_monitor.dataset_format import DatasetFormat

sm_session = sagemaker.Session(boto_session=boto3.Session(
    aws_access_key_id=AWS_CREDENTIALS["aws_access_key_id"],
    aws_secret_access_key=AWS_CREDENTIALS["aws_secret_access_key"],
    aws_session_token=AWS_CREDENTIALS.get("aws_session_token"),
    region_name=REGION,
))

model_monitor = DefaultModelMonitor(
    role=MONITOR_ROLE_ARN,
    instance_count=1,
    instance_type="ml.m5.xlarge",
    volume_size_in_gb=20,
    max_runtime_in_seconds=1800,
    sagemaker_session=sm_session,
)

# YOUR CODE: call model_monitor.suggest_baseline(
#     baseline_dataset=f"s3://{BUCKET_NAME}/input/training.csv",
#     dataset_format=DatasetFormat.csv(header=True),
#     output_s3_uri=f"s3://{BUCKET_NAME}/baseline-output/",
#     job_name=BASELINE_JOB,
#     wait=True,
# ).

print(f"Baseline job {BASELINE_JOB} submitted.")
print("Baseline is asking Deequ to read the data, give it about 5 minutes...")
elapsed = 0
status = None
while elapsed < 1200:
    try:
        d = sm.describe_processing_job(ProcessingJobName=BASELINE_JOB)
        status = d["ProcessingJobStatus"]
        if status in ("Completed", "Failed", "Stopped"):
            break
    except ClientError:
        pass
    time.sleep(15)
    elapsed += 15
    if elapsed % 60 == 0:
        print(f"  {elapsed}s elapsed, status: {status}")

if status != "Completed":
    print(f"Baseline job ended with status {status}.")
    raise SystemExit(1)
print("Baseline complete. Verifying statistics.json + constraints.json exist...")
for name in ("statistics.json", "constraints.json"):
    s3.head_object(Bucket=BUCKET_NAME, Key=f"baseline-output/{name}")
    print(f"  baseline-output/{name} present.")

In [ ]:
# NBVAL_SKIP
# Checkpoint 2: baseline job Completed and the two JSON files exist.

def checkpoint_2(session):
    try:
        sm_c = boto3.client(
            "sagemaker",
            aws_access_key_id=AWS_CREDENTIALS["aws_access_key_id"],
            aws_secret_access_key=AWS_CREDENTIALS["aws_secret_access_key"],
            aws_session_token=AWS_CREDENTIALS.get("aws_session_token"),
            region_name=REGION,
        )
        s3_c = boto3.client(
            "s3",
            aws_access_key_id=AWS_CREDENTIALS["aws_access_key_id"],
            aws_secret_access_key=AWS_CREDENTIALS["aws_secret_access_key"],
            aws_session_token=AWS_CREDENTIALS.get("aws_session_token"),
            region_name=REGION,
        )
        d = sm_c.describe_processing_job(ProcessingJobName=BASELINE_JOB)
        if d["ProcessingJobStatus"] != "Completed":
            return CheckpointResult(
                status="fail",
                error_reason=f"Baseline job status is {d['ProcessingJobStatus']!r}, not Completed.",
            )
        for name in ("statistics.json", "constraints.json"):
            try:
                s3_c.head_object(Bucket=BUCKET_NAME, Key=f"baseline-output/{name}")
            except ClientError:
                return CheckpointResult(
                    status="fail",
                    error_reason=(
                        f"baseline-output/{name} not found in S3. The Deequ baseline "
                        f"writer should produce both files; investigate the Processing "
                        f"Job logs in CloudWatch."
                    ),
                )
        return CheckpointResult(status="pass")
    except Exception as e:
        return CheckpointResult(status="error", error_reason=str(e))


check(2, checkpoint_2)

<details><summary>Hint 1 (nudge)</summary>

The `model_monitor` object is built. `suggest_baseline` is the call that runs the Deequ Processing Job; it takes the input CSV, the output prefix, and the dataset format.

</details>

<details><summary>Hint 2 (direction)</summary>

Pass `baseline_dataset` (S3 URI of training.csv), `dataset_format=DatasetFormat.csv(header=True)`, `output_s3_uri` (where statistics + constraints land), `job_name=BASELINE_JOB`, and `wait=True` (the SDK polls for you).

</details>

<details><summary>Hint 3 (spoiler)</summary>

Complete working solution for Task 2 (the suggest_baseline call):

```python
model_monitor.suggest_baseline(
    baseline_dataset=f"s3://{BUCKET_NAME}/input/training.csv",
    dataset_format=DatasetFormat.csv(header=True),
    output_s3_uri=f"s3://{BUCKET_NAME}/baseline-output/",
    job_name=BASELINE_JOB,
    wait=True,
)
```

</details>

**Wallet check.** Baseline Processing Job on ml.m5.xlarge at $0.230/hour for 5 minutes is about $0.02. The endpoint continues to bill in the background at $0.07/hour. Total damage so far: about $0.05. Coffee, still ahead.

## Task 3: Create the Monitoring Schedule wired to the endpoint, baseline, and a 15-minute cron expression

The Monitoring Schedule references the endpoint (via `EndpointInput`), the baseline (via `StatisticsResource` + `ConstraintsResource` pointing at the baseline-output prefix), and a cron schedule. The lab uses `cron(0/15 * * * ? *)` so the first run fires within 15 minutes instead of waiting an hour.

Call `model_monitor.create_monitoring_schedule(...)` with the endpoint input, the baseline URIs, the output prefix, and the schedule expression.

In [ ]:
# NBVAL_SKIP
# Task 3: Monitoring Schedule.

from sagemaker.model_monitor import EndpointInput, CronExpressionGenerator

endpoint_input = EndpointInput(
    endpoint_name=ENDPOINT_NAME,
    destination="/opt/ml/processing/endpointdata",
)

statistics_s3 = f"s3://{BUCKET_NAME}/baseline-output/statistics.json"
constraints_s3 = f"s3://{BUCKET_NAME}/baseline-output/constraints.json"
monitor_output_s3 = f"s3://{BUCKET_NAME}/monitor-output/"

# 15-minute cron (SageMaker accepts shorter intervals for testing).
schedule_expression = "cron(0/15 * * * ? *)"

# YOUR CODE: call model_monitor.create_monitoring_schedule(
#     monitor_schedule_name=SCHEDULE_NAME,
#     endpoint_input=endpoint_input,
#     statistics=statistics_s3,
#     constraints=constraints_s3,
#     output_s3_uri=monitor_output_s3,
#     schedule_cron_expression=schedule_expression,
#     enable_cloudwatch_metrics=True,
# ).

print(f"Monitoring schedule {SCHEDULE_NAME} created.")

print("Waiting 10 seconds for the schedule to register...")
time.sleep(10)
d = sm.describe_monitoring_schedule(MonitoringScheduleName=SCHEDULE_NAME)
print(f"Schedule status: {d['MonitoringScheduleStatus']}")
print(f"Schedule expression: {d['MonitoringScheduleConfig']['ScheduleConfig']['ScheduleExpression']}")

In [ ]:
# NBVAL_SKIP
# Checkpoint 3: schedule exists with the right baseline URIs and cron.

def checkpoint_3(session):
    try:
        sm_c = boto3.client(
            "sagemaker",
            aws_access_key_id=AWS_CREDENTIALS["aws_access_key_id"],
            aws_secret_access_key=AWS_CREDENTIALS["aws_secret_access_key"],
            aws_session_token=AWS_CREDENTIALS.get("aws_session_token"),
            region_name=REGION,
        )
        try:
            d = sm_c.describe_monitoring_schedule(MonitoringScheduleName=SCHEDULE_NAME)
        except ClientError as e:
            return CheckpointResult(
                status="fail",
                error_reason=f"Schedule {SCHEDULE_NAME} not found: {e}",
            )
        cfg = d.get("MonitoringScheduleConfig", {})
        cron = cfg.get("ScheduleConfig", {}).get("ScheduleExpression")
        if cron != "cron(0/15 * * * ? *)":
            return CheckpointResult(
                status="fail",
                error_reason=(
                    f"Schedule cron is {cron!r}, expected 'cron(0/15 * * * ? *)' "
                    f"(every 15 minutes, the lab pattern)."
                ),
            )
        baseline = (
            cfg.get("MonitoringJobDefinition", {})
            .get("BaselineConfig", {})
        )
        stats_uri = baseline.get("StatisticsResource", {}).get("S3Uri")
        cons_uri = baseline.get("ConstraintsResource", {}).get("S3Uri")
        if not stats_uri or "baseline-output" not in stats_uri:
            return CheckpointResult(
                status="fail",
                error_reason=(
                    f"Schedule StatisticsResource.S3Uri is {stats_uri!r}, expected to "
                    f"point at the baseline-output prefix."
                ),
            )
        if not cons_uri or "baseline-output" not in cons_uri:
            return CheckpointResult(
                status="fail",
                error_reason=(
                    f"Schedule ConstraintsResource.S3Uri is {cons_uri!r}, expected to "
                    f"point at the baseline-output prefix."
                ),
            )
        return CheckpointResult(status="pass")
    except Exception as e:
        return CheckpointResult(status="error", error_reason=str(e))


check(3, checkpoint_3)

<details><summary>Hint 1 (nudge)</summary>

The `model_monitor` SDK wraps the schedule API. `create_monitoring_schedule` takes the endpoint input, the baseline URIs, the output prefix, and the cron expression.

</details>

<details><summary>Hint 2 (direction)</summary>

Pass `monitor_schedule_name=SCHEDULE_NAME`, `endpoint_input=endpoint_input`, `statistics=statistics_s3`, `constraints=constraints_s3`, `output_s3_uri=monitor_output_s3`, `schedule_cron_expression=schedule_expression`, and `enable_cloudwatch_metrics=True`. No return value; the schedule registers server-side.

</details>

<details><summary>Hint 3 (spoiler)</summary>

Complete working solution for Task 3 (the create_monitoring_schedule call):

```python
model_monitor.create_monitoring_schedule(
    monitor_schedule_name=SCHEDULE_NAME,
    endpoint_input=endpoint_input,
    statistics=statistics_s3,
    constraints=constraints_s3,
    output_s3_uri=monitor_output_s3,
    schedule_cron_expression=schedule_expression,
    enable_cloudwatch_metrics=True,
)
```

</details>

**Wallet check.** Creating a monitoring schedule is free; you pay only for the Processing Jobs it fires. The endpoint at $0.07/hour continues to tick. Damage so far: about $0.06. Coffee still ahead.

## Task 4: Generate 100 baseline-distribution inferences and 100 drifted-distribution inferences against the endpoint

Inferences land in the DataCapture prefix as JSONL files (one per minute or per ~256 KB whichever fires first). The first 100 inferences use the baseline distribution (gaussian with mean 0 and stddev 1 across all 8 features). The next 100 use a deliberately drifted distribution: feature index 1 is shifted by 3 standard deviations (mean=3.0 instead of 0). All other features stay baseline-like so the violation report cleanly identifies feature 1 as the drifted column.

Your job: write the loop body that invokes the endpoint with each row. Both helper functions (`gen_baseline_row` and `gen_drifted_row`) are pre-wired.

In [ ]:
# NBVAL_SKIP
# Task 4: 100 baseline + 100 drifted inferences.

def gen_baseline_row():
    return ",".join(f"{random.gauss(0, 1):.4f}" for _ in range(8))


def gen_drifted_row():
    feats = [random.gauss(0, 1) for _ in range(8)]
    # Shift feature 1 by 3 standard deviations.
    feats[DRIFTED_FEATURE_INDEX] = feats[DRIFTED_FEATURE_INDEX] + 3.0
    return ",".join(f"{v:.4f}" for v in feats)


random.seed(13)
print("Sending 100 baseline-distribution inferences...")
for i in range(100):
    body = gen_baseline_row().encode("utf-8")
    # YOUR CODE: call sm_runtime.invoke_endpoint(EndpointName=ENDPOINT_NAME,
    # ContentType="text/csv", Body=body) and read the body to drain the response.
    pass
print("Sending 100 drifted-distribution inferences (feature index 1 shifted +3 stddev)...")
for i in range(100):
    body = gen_drifted_row().encode("utf-8")
    # YOUR CODE: same call shape, with the drifted body.
    pass

print("DataCapture batches writes every ~30 seconds, give it 90 more seconds to flush...")
time.sleep(90)

# Walk capture prefix and confirm objects exist.
capture_count = 0
paginator = s3.get_paginator("list_objects_v2")
for page in paginator.paginate(Bucket=BUCKET_NAME, Prefix="data-capture/"):
    for obj in page.get("Contents", []):
        capture_count += 1
print(f"DataCapture prefix object count: {capture_count}")

In [ ]:
# NBVAL_SKIP
# Checkpoint 4: at least 2 capture objects (one per batch) AND the captured
# rows include drifted samples (median of feature 1 across recent samples
# is above 1.0, well above the baseline median of ~0).

def checkpoint_4(session):
    try:
        s3_c = boto3.client(
            "s3",
            aws_access_key_id=AWS_CREDENTIALS["aws_access_key_id"],
            aws_secret_access_key=AWS_CREDENTIALS["aws_secret_access_key"],
            aws_session_token=AWS_CREDENTIALS.get("aws_session_token"),
            region_name=REGION,
        )
        keys = []
        paginator = s3_c.get_paginator("list_objects_v2")
        for page in paginator.paginate(Bucket=BUCKET_NAME, Prefix="data-capture/"):
            for obj in page.get("Contents", []):
                keys.append(obj["Key"])
        if len(keys) < 2:
            return CheckpointResult(
                status="fail",
                error_reason=(
                    f"DataCapture prefix has {len(keys)} objects, expected at least 2. "
                    f"Wait another 30-60 seconds and re-run; capture batches every ~30s."
                ),
            )
        f1_values = []
        for k in keys[-3:]:
            body = s3_c.get_object(Bucket=BUCKET_NAME, Key=k)["Body"].read().decode("utf-8")
            for line in body.strip().splitlines():
                try:
                    rec = json.loads(line)
                    cap = rec.get("captureData", {}).get("endpointInput", {}).get("data", "")
                    if not cap:
                        continue
                    parts = cap.strip().split(",")
                    if len(parts) >= 8:
                        f1_values.append(float(parts[DRIFTED_FEATURE_INDEX]))
                except Exception:
                    continue
        if not f1_values:
            return CheckpointResult(
                status="fail",
                error_reason=(
                    "Could not parse any captured rows. Verify invocations sent text/csv "
                    "bodies with 8 comma-separated values."
                ),
            )
        # Look at the latest 100 captured values; the drifted half should pull the mean up.
        recent = f1_values[-100:]
        mean_recent = sum(recent) / len(recent)
        if mean_recent < 0.5:
            return CheckpointResult(
                status="fail",
                error_reason=(
                    f"Mean of feature {DRIFTED_FEATURE_INDEX} across recent captures is "
                    f"{mean_recent:.3f}, expected above 0.5 once 100 drifted rows are in. "
                    f"Confirm the drifted-row loop ran."
                ),
            )
        return CheckpointResult(status="pass")
    except Exception as e:
        return CheckpointResult(status="error", error_reason=str(e))


check(4, checkpoint_4)

<details><summary>Hint 1 (nudge)</summary>

The endpoint invocation pattern is the same as Task 1. `sm_runtime.invoke_endpoint` accepts a CSV body; read the response body to drain it. The loop variable is `body` for each row.

</details>

<details><summary>Hint 2 (direction)</summary>

Each iteration calls `sm_runtime.invoke_endpoint(EndpointName=ENDPOINT_NAME, ContentType="text/csv", Body=body)["Body"].read()`. The read is required to consume the response so the connection is released back to the pool.

</details>

<details><summary>Hint 3 (spoiler)</summary>

Complete working solution for Task 4 (the invocation line that goes inside both loops):

```python
sm_runtime.invoke_endpoint(
    EndpointName=ENDPOINT_NAME,
    ContentType="text/csv",
    Body=body,
)["Body"].read()
```

</details>

**Wallet check.** 200 endpoint invocations against ml.t2.medium are essentially free; the endpoint bills the same per-hour rate regardless of throughput. The capture writes to S3 are fractions of a penny. Damage so far: about $0.08 (mostly the endpoint at $0.07/hr accruing). Coffee, comfortable.

## Task 5: Wait for the first scheduled monitoring run to complete and confirm it surfaced a violation on the drifted feature

This is the observe phase of the `code_observe` interaction mode. The 15-minute cron expression means the first execution fires within 15 minutes of schedule creation. Wait for `list_monitoring_executions` to return an execution with `MonitoringExecutionStatus=CompletedWithViolations`. Read the violation report from the monitor-output prefix and extract the drifted-feature name.

Note: `Completed` (no violations) means the drift was NOT detected, which is the failure mode. `CompletedWithViolations` is the lab's success path.

In [ ]:
# NBVAL_SKIP
# Task 5: observe phase. Wait for the first scheduled run, read violations.

print("Waiting for the first scheduled monitoring run to complete...")
print("Schedule fires every 15 minutes; the first run can be 5 to 20 minutes out.")

violation_record = None
elapsed = 0
while elapsed < 1500:  # 25-minute ceiling
    execs = sm.list_monitoring_executions(
        MonitoringScheduleName=SCHEDULE_NAME, MaxResults=10
    ).get("MonitoringExecutionSummaries", [])
    completed = [e for e in execs if e["MonitoringExecutionStatus"] in (
        "CompletedWithViolations", "Completed", "Failed"
    )]
    if completed:
        run = completed[0]
        print(f"\nFirst monitoring execution status: {run['MonitoringExecutionStatus']}")
        proc_arn = run.get("ProcessingJobArn")
        if proc_arn:
            job_name = proc_arn.split("/")[-1]
            proc = sm.describe_processing_job(ProcessingJobName=job_name)
            for out in proc.get("ProcessingOutputConfig", {}).get("Outputs", []):
                uri = out["S3Output"]["S3Uri"]
                bucket_part, _, key_part = uri.replace("s3://", "", 1).partition("/")
                # constraint_violations.json sits at the top of this prefix.
                try:
                    obj = s3.get_object(
                        Bucket=bucket_part,
                        Key=f"{key_part.rstrip('/')}/constraint_violations.json",
                    )
                    violation_record = json.loads(obj["Body"].read().decode("utf-8"))
                    break
                except ClientError:
                    continue
        break
    time.sleep(30)
    elapsed += 30
    if elapsed % 120 == 0:
        print(f"  {elapsed}s elapsed, still waiting for first scheduled execution...")

if violation_record is None:
    print("No monitoring execution completed within the 25-minute ceiling.")
    print("Inspect the SageMaker Studio Monitor view; the schedule may have skipped its window.")
    raise SystemExit(1)

violations = violation_record.get("violations", [])
drifted_features = [v.get("feature_name") for v in violations]
VIOLATION_COUNT = len(violations)
print(f"Violation count: {VIOLATION_COUNT}")
for v in violations[:5]:
    print(f"  feature={v.get('feature_name')!r} type={v.get('constraint_check_type')!r}")

In [ ]:
# NBVAL_SKIP
# Checkpoint 5: monitoring run produced at least one violation. The lab's
# comparisonMetric (rendered by the Option D Colab card) is the drifted
# feature name plus the violation count.

def checkpoint_5(session):
    try:
        sm_c = boto3.client(
            "sagemaker",
            aws_access_key_id=AWS_CREDENTIALS["aws_access_key_id"],
            aws_secret_access_key=AWS_CREDENTIALS["aws_secret_access_key"],
            aws_session_token=AWS_CREDENTIALS.get("aws_session_token"),
            region_name=REGION,
        )
        execs = sm_c.list_monitoring_executions(
            MonitoringScheduleName=SCHEDULE_NAME, MaxResults=10
        ).get("MonitoringExecutionSummaries", [])
        completed = [
            e for e in execs
            if e["MonitoringExecutionStatus"] == "CompletedWithViolations"
        ]
        if not completed:
            return CheckpointResult(
                status="fail",
                error_reason=(
                    "No execution with status 'CompletedWithViolations'. The drifted "
                    "payload should have produced one; check the latest execution status "
                    "in SageMaker Studio."
                ),
            )
        if VIOLATION_COUNT < 1:
            return CheckpointResult(
                status="fail",
                error_reason=(
                    "Violation count is 0 even though execution status is "
                    "CompletedWithViolations. The constraint_violations.json file is empty."
                ),
            )
        return CheckpointResult(status="pass")
    except Exception as e:
        return CheckpointResult(status="error", error_reason=str(e))


check(5, checkpoint_5)

print(
    f"comparisonMetric: Drifted feature: {DRIFTED_FEATURE_NAME}; "
    f"violation count: {VIOLATION_COUNT}"
)

<details><summary>Hint 1 (nudge)</summary>

The polling loop is wired. If no execution arrives in 25 minutes, check `list_monitoring_executions` directly in SageMaker Studio to verify the cron is firing.

</details>

<details><summary>Hint 2 (direction)</summary>

When the lab times out, the most common cause is a missed first window. Cron expressions are evaluated against UTC; the schedule fires the next 15-minute boundary (00, 15, 30, 45). If you created the schedule at :44 you wait until :45 plus the run duration (5 to 10 minutes). When the wait exceeds 25 minutes, re-run this cell; it picks up the latest execution.

</details>

<details><summary>Hint 3 (spoiler)</summary>

There is no code blank in this task; it is the observe phase of the `code_observe` mode. If the violation report failed to download, the manual command is:

```python
execs = sm.list_monitoring_executions(
    MonitoringScheduleName=SCHEDULE_NAME, MaxResults=10
)["MonitoringExecutionSummaries"]
for e in execs:
    if e["MonitoringExecutionStatus"] == "CompletedWithViolations":
        proc_arn = e["ProcessingJobArn"]
        print("Inspect processing job:", proc_arn)
        break
```

</details>

**Wallet check.** First scheduled monitoring run on ml.m5.xlarge for ~5 minutes adds about $0.02. The endpoint at $0.07/hour continues to tick. Total damage so far: about $0.12. Coffee is still ahead, but the cleanup cell is about to come for the endpoint.

In [ ]:
# NBVAL_SKIP
# Cleanup. The cleanup order matters: stop and delete the monitoring
# schedule first, then the endpoint. If the schedule outlives the endpoint
# it fires Processing Jobs against an endpoint that does not exist anymore,
# at $0.02 per misfire.

sm_cleanup = boto3.client(
    "sagemaker",
    aws_access_key_id=AWS_CREDENTIALS["aws_access_key_id"],
    aws_secret_access_key=AWS_CREDENTIALS["aws_secret_access_key"],
    aws_session_token=AWS_CREDENTIALS.get("aws_session_token"),
    region_name=REGION,
)

manual_warnings = []


def _safe(label, action, fallback_cmd):
    try:
        action()
        print(f"Deleted {label}.")
    except ClientError as e:
        code = e.response["Error"]["Code"]
        if code in ("ValidationException", "ResourceNotFound", "ResourceNotFoundException"):
            print(f"{label} already gone, skipping.")
        else:
            manual_warnings.append(
                f"FAILED TO DELETE: {label}. Error: {e}. Run manually: {fallback_cmd}"
            )


# 1. Stop the monitoring schedule (required before delete).
schedule_stopped = False
try:
    sm_cleanup.stop_monitoring_schedule(MonitoringScheduleName=SCHEDULE_NAME)
    print("Requested stop on monitoring schedule.")
    elapsed = 0
    while elapsed < 180:
        d = sm_cleanup.describe_monitoring_schedule(MonitoringScheduleName=SCHEDULE_NAME)
        if d["MonitoringScheduleStatus"] in ("Stopped", "Failed"):
            schedule_stopped = True
            break
        time.sleep(10)
        elapsed += 10
except ClientError as e:
    if e.response["Error"]["Code"] in ("ValidationException", "ResourceNotFound"):
        schedule_stopped = True
        print("Schedule already gone.")
    else:
        manual_warnings.append(f"FAILED to stop schedule: {e}")

# 2. Delete schedule.
_safe(
    f"monitoring schedule {SCHEDULE_NAME}",
    lambda: sm_cleanup.delete_monitoring_schedule(MonitoringScheduleName=SCHEDULE_NAME),
    f"aws sagemaker delete-monitoring-schedule --monitoring-schedule-name {SCHEDULE_NAME}",
)

# 3. Delete endpoint (CRITICAL: $0.07/hour stops here).
endpoint_gone = False
_safe(
    f"endpoint {ENDPOINT_NAME}",
    lambda: sm_cleanup.delete_endpoint(EndpointName=ENDPOINT_NAME),
    f"aws sagemaker delete-endpoint --endpoint-name {ENDPOINT_NAME}",
)
print("Waiting for endpoint to disappear...")
elapsed = 0
while elapsed < 300:
    try:
        sm_cleanup.describe_endpoint(EndpointName=ENDPOINT_NAME)
        time.sleep(10)
        elapsed += 10
    except ClientError as e:
        if e.response["Error"]["Code"] in ("ValidationException", "ResourceNotFound"):
            endpoint_gone = True
            break
        raise
if not endpoint_gone:
    manual_warnings.append(
        f"Endpoint {ENDPOINT_NAME} did not disappear within 5 minutes. Run: "
        f"aws sagemaker delete-endpoint --endpoint-name {ENDPOINT_NAME}"
    )

# 4. Delete endpoint config and model.
_safe(
    f"endpoint config {ENDPOINT_CONFIG_NAME}",
    lambda: sm_cleanup.delete_endpoint_config(EndpointConfigName=ENDPOINT_CONFIG_NAME),
    f"aws sagemaker delete-endpoint-config --endpoint-config-name {ENDPOINT_CONFIG_NAME}",
)
_safe(
    f"model {MODEL_NAME}",
    lambda: sm_cleanup.delete_model(ModelName=MODEL_NAME),
    f"aws sagemaker delete-model --model-name {MODEL_NAME}",
)

# 5. Hand off to run_cleanup for IAM role + S3 bucket.
result = run_cleanup(CLEANUP_MANIFEST)

for warning in manual_warnings:
    print(warning)
for warning in result.warnings:
    print(warning)
for orphan in result.orphans:
    print(orphan)

failed_ids = set()
for w in result.warnings:
    for r in CLEANUP_MANIFEST:
        if r.id in w:
            failed_ids.add(r.id)
            break

critical_total = 2  # endpoint + schedule
critical_destroyed = (1 if endpoint_gone else 0) + (1 if schedule_stopped else 0)
standard_total = len(CLEANUP_MANIFEST)
standard_destroyed = standard_total - len(failed_ids)
failed_count = len(failed_ids) + len(manual_warnings)

print("Cleanup complete.")
print(f"Critical resources destroyed: {critical_destroyed} of {critical_total}")
print(f"Standard resources destroyed: {standard_destroyed} of {standard_total}")
print(f"Resources that failed to delete: {failed_count} (see above for CLI commands)")
print("If K > 0, your AWS account may still incur charges. Resolve before closing this notebook.")

final_status = "clean" if (
    failed_count == 0
    and result.status == "clean"
    and endpoint_gone
    and schedule_stopped
) else "dirty"
cleanup(status=final_status)

if failed_count > 0 or not (endpoint_gone and schedule_stopped):
    sys.exit(1)

**Session total: about $0.10 to $0.20.** Real-Time endpoint on ml.t2.medium at $0.07 per hour for 60 to 90 minutes is the line item. The two Processing Jobs (baseline + first scheduled monitor) on ml.m5.xlarge for 5 minutes each add about 4 cents. DataCapture S3 puts are fractions of a penny. CloudWatch metrics are free at this volume. If you confirmed in the cleanup summary that the endpoint and schedule are both gone, the $0.07/hour bill stopped. Check your AWS Billing console in 24 hours to confirm zero ongoing charges from this lab.

## Reflection

These are not graded. They are for you.

1. Walk through the four Model Monitor types SageMaker supports (Data Quality, Model Quality, Bias Drift, Feature Attribution Drift). For each one, write one sentence on what it monitors and one sentence on when you would pick it. Why is Data Quality the one this lab uses, and what additional ground-truth-required workflow does Model Quality assume that Data Quality does not?

2. The monitoring schedule fires every 15 minutes in this lab to keep the demo tight, but production cadences are typically hourly or daily. Walk through the cost vs detection-latency trade-off: more frequent runs catch drift faster but cost more in Processing Job compute. Where does a daily cadence fit (slow-moving features), where does hourly fit (fraud, ads), and where does the team need a real-time streaming-detection pattern entirely outside Model Monitor?

## Exam-Style Practice

**Question 1 (Domain 4, Model Monitor type selection):**

A team needs production drift detection for: (a) a credit-scoring model whose training feature distributions might shift quarterly as the macroeconomy changes, and (b) a churn model whose predictions can be ground-truth-labeled 30 days later when customer behavior is observed. Which combination of Model Monitor types fits each requirement?

A. Data Quality monitor for the credit-scoring feature distribution; Model Quality monitor for the churn-prediction-vs-ground-truth comparison.

B. Bias Drift monitor for both, because Bias Drift is the unified production drift tool.

C. Data Quality monitor for both, because feature drift is the only drift type worth tracking.

D. Feature Attribution Drift monitor for both, because SHAP attribution captures all drift patterns.

<details><summary>Show answer</summary>

**Correct: A.**

- A is correct: Data Quality monitors compare incoming feature distributions against a baseline statistics.json file; they fit the credit-scoring use case where the team wants to detect when input feature distributions drift over time. Model Quality monitors compare predicted vs actual (ground-truth) labels and require a ground-truth ingestion path; they fit the churn use case where the team has a 30-day ground-truth label and wants to detect model-performance degradation.
- B is wrong: Bias Drift monitors track statistical-fairness metrics across protected facets over time. They are not the canonical 'feature distribution drift' or 'prediction accuracy drift' monitor for the two scenarios in the question.
- C is wrong: Data Quality alone cannot detect a model whose predictions are diverging from actual outcomes; that requires Model Quality with ground truth.
- D is wrong: Feature Attribution Drift monitors track per-feature SHAP attribution changes over time. It is a complementary tool, not the right primary fit for either scenario.

</details>

**Question 2 (Domain 4, Model Monitor cleanup ordering):**

A team set up a SageMaker Monitoring Schedule on an endpoint at `cron(0 * * * ? *)` (hourly). Six weeks later they deleted the endpoint via the Console to save cost. The next month their bill shows roughly $30 in SageMaker Processing charges. Investigation reveals the monitoring schedule has fired 1,080 times (24 x 45 days), each time failing with 'endpoint not found' but still spinning up the Processing Job. What is the AWS-recommended cleanup order, and how should the team prevent this in the future?

A. Delete the monitoring schedule first via `delete_monitoring_schedule`, then delete the endpoint. To prevent recurrence, always tear down monitoring schedules before endpoints.

B. Stop the monitoring schedule with `stop_monitoring_schedule`, then delete it with `delete_monitoring_schedule`, then delete the endpoint. Stopping a live schedule on a deleted endpoint requires the stop call before delete because AWS does not accept delete on a Scheduled schedule.

C. CloudWatch will auto-detect orphan monitoring schedules within 24 hours.

D. The team can ignore this because failed Processing Jobs do not bill for compute time.

<details><summary>Show answer</summary>

**Correct: B.**

- A is partially right on the ordering (schedule before endpoint) but missing the stop step. `delete_monitoring_schedule` on a Scheduled schedule fails with ResourceInUse.
- B is correct: the AWS-published teardown order for monitoring is `stop_monitoring_schedule` then poll until Stopped status then `delete_monitoring_schedule` then `delete_endpoint`. Failed Processing Jobs DO bill for the compute they consumed before failing (typically 60-90 seconds at the configured instance rate).
- C is wrong: CloudWatch does not auto-cleanup orphan SageMaker resources. There is no AWS-provided cleanup automation for this scenario.
- D is wrong: failed Processing Jobs bill for compute time. The team's $30 charge is real.

</details>